DATASET

In [ ]:
import pandas as pd

data = [
    ["young","myope","no","reduced","none"],
    ["young","myope","no","normal","soft"],
    ["young","myope","yes","reduced","none"],
    ["young","myope","yes","normal","hard"],
    ["young","hypermetrope","no","reduced","none"],
    ["young","hypermetrope","no","normal","soft"],
    ["young","hypermetrope","yes","reduced","none"],
    ["young","hypermetrope","yes","normal","hard"],

    ["pre-presbyopic","myope","no","reduced","none"],
    ["pre-presbyopic","myope","no","normal","soft"],
    ["pre-presbyopic","myope","yes","reduced","none"],
    ["pre-presbyopic","myope","yes","normal","hard"],
    ["pre-presbyopic","hypermetrope","no","reduced","none"],
    ["pre-presbyopic","hypermetrope","no","normal","soft"],
    ["pre-presbyopic","hypermetrope","yes","reduced","none"],
    ["pre-presbyopic","hypermetrope","yes","normal","none"],

    ["presbyopic","myope","no","reduced","none"],
    ["presbyopic","myope","no","normal","none"],
    ["presbyopic","myope","yes","reduced","none"],
    ["presbyopic","myope","yes","normal","hard"],
    ["presbyopic","hypermetrope","no","reduced","none"],
    ["presbyopic","hypermetrope","no","normal","soft"],
    ["presbyopic","hypermetrope","yes","reduced","none"],
    ["presbyopic","hypermetrope","yes","normal","none"],
]

df = pd.DataFrame(data, columns=[
    "age", "spectacle", "astigmatism", "tear", "lenses"
])

df

,age,spectacle,astigmatism,tear,lenses
0,young,myope,no,reduced,none
1,young,myope,no,normal,soft
2,young,myope,yes,reduced,none
3,young,myope,yes,normal,hard
4,young,hypermetrope,no,reduced,none
5,young,hypermetrope,no,normal,soft
6,young,hypermetrope,yes,reduced,none
7,young,hypermetrope,yes,normal,hard
8,pre-presbyopic,myope,no,reduced,none
9,pre-presbyopic,myope,no,normal,soft


ESCOLHER CLASSE

In [ ]:
classe_alvo = "none"  # troque para: "hard" ou "none"

FUNÇÃO DE PROBABILIDADE

In [ ]:
def calcular_probabilidades(df, target):
    resultados = []
    atributos = ["age", "spectacle", "astigmatism", "tear"]

    for atributo in atributos:
        for valor in df[atributo].unique():
            subset = df[df[atributo] == valor]

            total = len(subset)
            positivos = len(subset[subset["lenses"] == target])

            if total > 0:
                prob = positivos / total
            else:
                prob = 0

            resultados.append({
                "atributo": atributo,
                "valor": valor,
                "prob": prob,
                "positivos": positivos,
                "total": total
            })

    resultado = pd.DataFrame(resultados)

    return resultado.sort_values(
        by=["prob", "positivos"],
        ascending=[False, False]
    )

EXECUTAR INTERAÇÕES - caso tenha mais de uma interação, faz o loop

In [ ]:
df_atual = df.copy()
regra = []

while True:
    print("\n==============================")
    print("Subconjunto atual:")
    display(df_atual)

    probs = calcular_probabilidades(df_atual, classe_alvo)

    # remover candidatos sem positivos
    probs = probs[probs["positivos"] > 0]

    print("\nProbabilidades:")
    display(probs)

    # escolher melhor
    melhor = probs.iloc[0]

    print("\nEscolhido:")
    print(melhor)

    regra.append((melhor["atributo"], melhor["valor"]))

    # filtrar
    df_atual = df_atual[
        df_atual[melhor["atributo"]] == melhor["valor"]
    ]

    # verificar parada
    if len(df_atual["lenses"].unique()) == 1:
        print("\n✅ Regra ficou perfeita (100%)")
        break

    # segurança (sem mais atributos úteis)
    if len(probs) == 0:
        print("\n⚠️ Não há mais refinamento possível")
        break


Subconjunto atual:


,age,spectacle,astigmatism,tear,lenses
0,young,myope,no,reduced,none
1,young,myope,no,normal,soft
2,young,myope,yes,reduced,none
3,young,myope,yes,normal,hard
4,young,hypermetrope,no,reduced,none
5,young,hypermetrope,no,normal,soft
6,young,hypermetrope,yes,reduced,none
7,young,hypermetrope,yes,normal,hard
8,pre-presbyopic,myope,no,reduced,none
9,pre-presbyopic,myope,no,normal,soft



Probabilidades:


,atributo,valor,prob,positivos,total
7,tear,reduced,1.000000,12,12
2,age,presbyopic,0.750000,6,8
4,spectacle,hypermetrope,0.666667,8,12
6,astigmatism,yes,0.666667,8,12
1,age,pre-presbyopic,0.625000,5,8
3,spectacle,myope,0.583333,7,12
5,astigmatism,no,0.583333,7,12
0,age,young,0.500000,4,8
8,tear,normal,0.250000,3,12



Escolhido:
atributo        tear
valor        reduced
prob             1.0
positivos         12
total             12
Name: 7, dtype: object

✅ Regra ficou perfeita (100%)


MOSTRAR REGRA FINAL

In [ ]:
condicoes = " AND ".join(
    [f"{attr} = {val}" for attr, val in regra]
)

print("\n🎯 REGRA FINAL:")
print(f"IF {condicoes} THEN lenses = {classe_alvo}")


🎯 REGRA FINAL:
IF tear = reduced THEN lenses = none
